# ASE / ACE

This is the ASE / ACE architecture for cartpole from [Sutton & Barto's 1983 paper, Neuronlike adaptive elements that can solve difficult learning control problems](https://ieeexplore.ieee.org/document/6313077), with modern tools (OpenAI's Gymnasium + Numpy / Jax).

First of all here is my collaged notes on how their system works:

<img src="../img/suttonbarto_notes.jpg" width="500" />

And here is my super scratchy pseudocode (which I ended up deviating from a bit):

<img src="../img/suttonbarto_pseudocode.jpg" width="300" />

In [2]:
import numpy as np
import jax
import jax.numpy as jnp

N_BOXES = 162

def decoder(observation):
    """
    takes in state (x, x_dot, theta, theta_dot) and discretizes into 162 bins as specified in the paper, 
    from Michie & Chambers' boxes system
    """
    x, x_dot, theta, theta_dot = observation

    # (3) x: +/- 0.8, +/- 2.4
    if x < -0.8:
        x_bin = 0
    elif x < 0.8:
        x_bin = 1
    else:
        x_bin = 2
    
    # (6) theta: 0, +/- 1, +/- 6, +/- 12
    theta_deg = jnp.degrees(theta)  # paper specifies thresholds in degrees
    if theta_deg < -6:
        theta_bin = 0
    elif theta_deg < -1:
        theta_bin = 1
    elif theta_deg < 0:
        theta_bin = 2
    elif theta_deg < 1:
        theta_bin = 3
    elif theta_deg < 6:
        theta_bin = 4
    else:
        theta_bin = 5
    
    # (3) x_dot: +/- 0.5, +/- inf
    if x_dot < -0.5:
        x_dot_bin = 0
    elif x_dot < 0.5:
        x_dot_bin = 1
    else:
        x_dot_bin = 2

    # (3) theta_dot: +/0 50, +/- inf
    theta_dot_deg = jnp.degrees(theta_dot)
    if theta_dot_deg < -50:
        theta_dot_bin = 0
    elif theta_dot_deg < 50:
        theta_dot_bin = 1
    else:
        theta_dot_bin = 2

    # convert into idx of 3 * 6 * 3 * 3 = 162 bins
    box_idx = (
        x_bin * (3 * 6 * 3)
        + x_dot_bin * (6 * 3)
        + theta_bin * 3
        + theta_dot_bin
    )
    arr = jnp.zeros(N_BOXES)
    arr = arr.at[box_idx].set(1)
    return arr

In [3]:
from typing import NamedTuple

class ASEState(NamedTuple):
    """
    Associative Search Element;
    produces control action from box vector + reinforcement signal (external or internal)
    """
    w: jnp.ndarray # (n_boxes,)
    e: jnp.ndarray # (n_boxes,)

def ase_init(n_boxes):
    """
    initialize an all-zero starting state for ASE
    """
    return ASEState(w=jnp.zeros(n_boxes), e=jnp.zeros(n_boxes))

def ase_reset_traces(state):
    """
    run between trials - 
    reset e, but keep v (so we keep learning!)
    """
    return state._replace(
        e=jnp.zeros_like(state.e),
    )

def ase_get_action(state, box_vec, seed, sigma):
    """
    takes current ASE state, box_vec for our current obs, random seed, and sigma (std of noise)
    returns action as per eqn (1)
    """
    noise = jax.random.normal(seed) * sigma
    inner = state.w @ box_vec + noise
    # f(x)
    return jnp.where(inner >= 0, 1.0, -1.0)  # 1 if inner is >= 0, otherwise -1

def ase_update(state, last_x, last_y, r_hat, alpha, delta):
    """
    takes ASE state, previous x, previous y, internal reward, and hyperparams alpha, delta
    updates ASE weights as per eqns (2), (3)
    """
    w_new = state.w + alpha * r_hat * state.e # eqn (2)
    e_new = delta * state.e + (1 - delta) * last_y * last_x # eqn (3)
    return ASEState(w=w_new, e=e_new)

In [4]:
class ACEState(NamedTuple):
    """
    Adaptive Critic Element;
    helps with credit assignment by computing internal reward from box vec + reinforcement signal (-1 if failure, 0 otherwise)
    """
    v: jnp.ndarray # (n_boxes,)
    x_bar: jnp.ndarray # (n_boxes,)
    p_prev: jnp.ndarray # scalar — p(t-1), previous prediction

def ace_init(n_boxes):
    """
    initialize an all-zero starting state for ACE 
    """
    return ACEState(
        v=jnp.zeros(n_boxes),
        x_bar=jnp.zeros(n_boxes),
        p_prev=jnp.array(0.0),
    )

def ace_reset_traces(state):
    """
    run between trials - 
    reset x_bar and p_prev, but keep v (so we keep learning!)
    """
    return state._replace(
        x_bar=jnp.zeros_like(state.x_bar),
        p_prev=jnp.array(0.0),
    )

def ace_get_r_hat(state, box_vec, r, gamma):
    """
    get r_hat from ACE; also returns p because we need it for the update
    """
    p = state.v @ box_vec # (4)
    r_hat = r + gamma * p - state.p_prev # (7)
    return r_hat, p


def ace_update(state, box_vec, r_hat, p, beta, lam):
    """
    update ACE params
    """
    v_new = state.v + beta * r_hat * state.x_bar # (5)

    x_bar_new = lam * state.x_bar + (1 - lam) * box_vec # (6)

    return ACEState(v=v_new, x_bar=x_bar_new, p_prev=p)

In [5]:
import gymnasium as gym
import jax
import jax.numpy as jnp
import numpy as np
from tqdm import tqdm

env = gym.make(
    "CartPole-v1",
    render_mode="none",
    sutton_barto_reward=True,
)

N_REPS = 10 # run the whole learning process 10 times, then take the average
N_TRIALS = 100 # number of episodes to learn from
N_BOXES = 162

# hyperparams from the paper
ALPHA = 1000.0
BETA = 0.5
DELTA = 0.9
LAM = 0.8
GAMMA = 0.95
SIGMA = 0.01

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    # init ase, ace state (persists across trials)
    ase_state = ase_init(N_BOXES)
    ace_state = ace_init(N_BOXES)

    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    trial_results = []

    pbar = tqdm(range(N_TRIALS), desc=f"repeat {rep}")
    for trial_idx in pbar:
        observation, info = env.reset()

        # Reset traces between trials, keep learned weights
        ase_state = ase_reset_traces(ase_state)
        ace_state = ace_reset_traces(ace_state)

        episode_over = False
        steps = 0

        while not episode_over:
            box_vec = decoder(observation)

            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # sample action
            action_pm = ase_get_action(ase_state, box_vec, act_key, SIGMA)  # +/- 1
            action_env = int((action_pm + 1) // 2)  # map {-1, +1} -> {0, 1} for gym

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action_env)
            steps += 1

            # get box for new obs
            # at failure the decoder returns all zeros (per the paper), so p_next = 0 automatically.
            box_vec_next = decoder(observation)

            # ACE: compute r_hat and update
            r_hat, p = ace_get_r_hat(ace_state, box_vec_next, reward, GAMMA)
            ace_state = ace_update(ace_state, box_vec_next, r_hat, p, BETA, LAM)

            # ASE: update using last (box_vec, action_pm) and r_hat from ACE
            ase_state = ase_update(ase_state, box_vec, action_pm, r_hat, ALPHA, DELTA)

            episode_over = terminated or truncated

        trial_results.append(steps)
        pbar.set_postfix({"steps": steps})

    pbar.close()
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)

# Compute average + std steps alive at the last (100th) trial across reps
final_trial_steps = all_trial_results[:, -1]  # shape (N_REPS,)
mean_final = final_trial_steps.mean()
std_final = final_trial_steps.std()
print(f"Final trial (#{N_TRIALS}) steps alive across {N_REPS} reps: "
      f"{mean_final:.2f} ± {std_final:.2f}")

/Users/miranda/miniconda3/envs/nmm_finalproject/lib/python3.12/site-packages/gymnasium/envs/registration.py:729: UserWarning: WARN: The environment is being initialised with render_mode='none' that is not in the possible render_modes (['human', 'rgb_array']).
  logger.warn(
repeat 9: 100%|██████████| 100/100 [00:14<00:00,  6.73it/s, steps=500]

Final trial (#100) steps alive across 10 reps: 238.00 ± 176.41


In [7]:
model_name = "ASE / ACE"

In [9]:
import plotly.graph_objects as go
import numpy as np

def plot_trial_results(model_name, mean_curve, std_curve=None):
    """Plot mean steps-until-failure per trial, with optional std band."""
    trials = np.arange(len(mean_curve))
    fig = go.Figure()
    title=f"{model_name} Learning Curve"

    if std_curve is not None:
        # Upper bound (invisible line, anchors the fill)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve + std_curve,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ))
        # Lower bound (fills down to upper bound)
        fig.add_trace(go.Scatter(
            x=trials,
            y=mean_curve - std_curve,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(0, 100, 200, 0.2)",
            name="±1 std",
            hoverinfo="skip",
        ))

    # Mean line on top
    fig.add_trace(go.Scatter(
        x=trials,
        y=mean_curve,
        mode="lines+markers",
        name="Mean steps until failure",
        line=dict(color="rgb(0, 100, 200)"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Trial",
        yaxis_title="Steps until failure",
        template="plotly_white",
    )
    fig.show()

mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)
plot_trial_results(model_name, mean_curve, std_curve)

# Thinking outside the BOXES (Michie & Chambers, 1968)
This performance is not nearly as good as their paper; possibly because the values they chose for the boxes were chosen for their particular simulation equations, which the Gymnasium environment may not be exactly replicating.

Plus, what they are basically doing is representation learning, though interestingly they're going from a low-dimensional state to a higher-dimensional latent representation. So, I'm wondering if we can apply our more modern throw-compute-at-it-and-don't-hardcode-things-yourself approach and dunk on our good friends Sutton and Barto.

(edit: turns out the answer is yes, this paper was an ancestor of modern actor-critic methods which do pretty much just this, but with some clever tricks. I think for my final project I will try to reproduce a bit of the lineage, and also compare this method to classical controls methods).